# 🔬 DME Fine-Tuning Pipeline

End-to-end notebook for training and evaluating the DME head of the multi-task
Diabetic Retinopathy + Diabetic Macular Edema detection system.

## Pipeline Overview
```
Raw Images ──► Preprocessing ──► tf.data pipeline ──► DME Head Training ──► Evaluation
```

**Training strategy (transfer learning):**
- Backbone (ResNet50): 🔒 Frozen
- ASPP block:          🔒 Frozen
- DR head:             🔒 Frozen
- DME head:            🟢 Trainable only

## 0. Setup & Imports

In [ ]:
import os
import json
import logging
import warnings

import numpy as np
import tensorflow as tf

# Suppress noisy logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Project modules
from preprocess import preprocess_image, load_and_preprocess
from dataset_loader import build_datasets, create_mock_dataset, DME_CLASSES
from model import build_model_dme_tuning, print_model_summary
from train import compile_dme_model, build_callbacks, DEFAULT_CONFIG
from evaluate import evaluate, DME_CLASS_NAMES

print("All modules imported successfully ✓")

## 1. Configuration

In [ ]:
# ── Dataset paths ──────────────────────────────────────────────────────────
# Update these paths for your environment:
#   Kaggle:   /kaggle/input/idrid/
#   Local:    data/IRDID/
#   Colab:    /content/drive/MyDrive/IRDID/

USE_MOCK_DATA = True          # Set False when real IRDID data is available

CSV_PATH  = "data/IRDID/DME_Grades.csv"
IMAGE_DIR = "data/IRDID/images"

# Pretrained backbone weights from Stage 0
PRETRAINED_WEIGHTS = "pretrain_final.weights.h5"  # None if not available

# ── Training hyper-parameters ──────────────────────────────────────────────
CONFIG = {
    **DEFAULT_CONFIG,
    "epochs": 30,
    "batch_size": 8,
    "learning_rate": 1e-4,
    "early_stopping_patience": 5,
    "checkpoint_dir": "checkpoints",
    "history_path": "training_history.json",
    "log_path": "training_log.txt",
}

# Output
OUTPUT_WEIGHTS  = "dme_finetuned.weights.h5"
RESULTS_DIR     = "results"

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:<35} {v}")

## 2. Dataset Loading & Preprocessing

In [ ]:
if USE_MOCK_DATA:
    print("⚠️  Using MOCK data (set USE_MOCK_DATA=False for real IRDID training)")
    CSV_PATH, IMAGE_DIR = create_mock_dataset(
        output_dir="/tmp/mock_irdid",
        num_samples=80,
    )

train_ds, val_ds, class_weights = build_datasets(
    csv_path=CSV_PATH,
    image_dir=IMAGE_DIR,
    target_size=(512, 512),
    batch_size=CONFIG["batch_size"],
    val_split=0.2,
    augment_train=True,
)

print(f"\nClass weights: {class_weights}")
print(f"Train batches : {len(train_ds)}")
print(f"Val   batches : {len(val_ds)}")

### 2.1 Visualise sample batch

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sample_images, sample_labels = next(iter(train_ds))
num_show = min(8, len(sample_images))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(num_show):
    img = sample_images[i].numpy()
    label_idx = int(np.argmax(sample_labels[i].numpy()))
    axes[i].imshow(img)
    axes[i].set_title(f"{DME_CLASS_NAMES[label_idx]} (class {label_idx})", fontsize=10)
    axes[i].axis("off")

plt.suptitle("Sample Training Batch (after preprocessing)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_batch.png", dpi=120, bbox_inches="tight")
plt.show()
print("Sample batch saved to sample_batch.png")

## 3. Build DME Fine-Tuning Model

In [ ]:
# Use pretrained weights if available, otherwise start from ImageNet
weights_path = PRETRAINED_WEIGHTS if os.path.exists(PRETRAINED_WEIGHTS) else None
if weights_path is None:
    print("ℹ️  pretrain_final.weights.h5 not found – initialising from ImageNet")

model = build_model_dme_tuning(
    input_shape=(512, 512, 3),
    pretrained_weights=weights_path,
    num_dme_classes=4,
)

print_model_summary(model)

In [ ]:
# Verify only DME layers are trainable
print("\nTrainable layers:")
for layer in model.layers:
    if layer.trainable:
        print(f"  ✅ {layer.name}")

## 4. Compile Model

In [ ]:
model = compile_dme_model(model, learning_rate=CONFIG["learning_rate"])
print("Model compiled ✓")

## 5. Training

In [ ]:
callbacks = build_callbacks(
    checkpoint_dir=CONFIG["checkpoint_dir"],
    history_path=CONFIG["history_path"],
    log_path=CONFIG["log_path"],
    early_stopping_patience=CONFIG["early_stopping_patience"],
    lr_reduce_patience=CONFIG["lr_reduce_patience"],
    lr_reduce_factor=CONFIG["lr_reduce_factor"],
    min_lr=CONFIG["min_lr"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs"],
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Save final weights
model.save_weights(OUTPUT_WEIGHTS)
print(f"Saved fine-tuned weights → {OUTPUT_WEIGHTS}")

## 6. Training History Visualisation

In [ ]:
hist = history.history

# Identify available DME metrics
loss_key   = "dme_risk_loss"     if "dme_risk_loss"     in hist else "loss"
acc_key    = "dme_risk_accuracy" if "dme_risk_accuracy" in hist else "accuracy"
val_loss   = "val_" + loss_key
val_acc    = "val_" + acc_key

epochs_ran = range(1, len(hist[loss_key]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_ran, hist[loss_key],  label="Train Loss",  linewidth=2)
ax1.plot(epochs_ran, hist[val_loss],  label="Val Loss",    linewidth=2, linestyle="--")
ax1.set_title("DME Loss", fontsize=13)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_ran, hist[acc_key],  label="Train Acc",  linewidth=2)
ax2.plot(epochs_ran, hist[val_acc],  label="Val Acc",    linewidth=2, linestyle="--")
ax2.set_title("DME Accuracy", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Training History", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("training_history.png", dpi=120, bbox_inches="tight")
plt.show()
print("Training history saved → training_history.png")

## 7. Comprehensive Evaluation

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

metrics = evaluate(
    model=model,
    dataset=val_ds,
    class_names=DME_CLASS_NAMES,
    output_dir=RESULTS_DIR,
    confusion_matrix_path="confusion_matrix.png",
    metrics_path="evaluation_metrics.json",
)

print("\n📊 Evaluation Results")
print(f"  Accuracy   : {metrics['accuracy']:.4f}")
print(f"  F1 (macro) : {metrics['f1_macro']:.4f}")
print(f"  F1 (weighted): {metrics['f1_weighted']:.4f}")
if metrics.get("roc_auc"):
    print(f"  ROC-AUC macro: {metrics['roc_auc'].get('macro', 'N/A'):.4f}")

In [ ]:
# Display per-class metrics
if metrics.get("per_class"):
    print("\nPer-class metrics:")
    print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 55)
    for cls, vals in metrics["per_class"].items():
        print(
            f"{cls:<12} {vals['precision']:>10.3f} {vals['recall']:>10.3f} "
            f"{vals['f1']:>10.3f} {vals['support']:>10d}"
        )

In [ ]:
# Show confusion matrix
from IPython.display import Image, display
cm_path = os.path.join(RESULTS_DIR, "confusion_matrix.png")
if os.path.exists(cm_path):
    display(Image(cm_path))

## 8. Summary

In [ ]:
print("=" * 60)
print("  TRAINING COMPLETE")
print("=" * 60)
print(f"  Weights saved   : {OUTPUT_WEIGHTS}")
print(f"  History JSON    : {CONFIG['history_path']}")
print(f"  Training CSV    : {CONFIG['log_path']}")
print(f"  Eval metrics    : {os.path.join(RESULTS_DIR, 'evaluation_metrics.json')}")
print(f"  Confusion matrix: {os.path.join(RESULTS_DIR, 'confusion_matrix.png')}")
print("=" * 60)

threshold_met = metrics.get("f1_macro", 0) >= 0.70
status = "✅ F1 ≥ 0.70 — Success!" if threshold_met else "⚠️  F1 < 0.70 — consider more epochs or data augmentation"
print(f"  {status}")